<a id="top"></a>

# Demo: tool, or no tool?

```yaml
title: "Demo: tool, or no tool?"
keywords: tool-or-no-tool, when to add a tool, model-alone, calculator, current_time, architectural decision, affordances, langchain, bind_tools, teacher demo
estimated duration: 12
```

> **Lesson:** L08. Teacher demo — Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies**: dry-run before class. Clear outputs before committing.
>
> **Model-agnostic by design.** L08 registers tools and watches the model *choose* and *call* them, so — like L07 — the demos drive a LangChain `ChatAnthropic` (swappable for `ChatOpenAI`). Each tool is a plain typed function; `bind_tools` derives its definition from the signature and docstring. The API key loads through the config seam (`require_anthropic_key`); we never hard-code it.
>
> The point to land: **adding a tool is an architectural decision, not a convenience.** The same task can be answered model-alone or with a tool — the right call depends on the *kind* of question. We run three tasks model-alone first, *then* with a tool bound, and name why.

## Contents

- [1. Setup — two clean tools and three tasks](#1-setup--two-clean-tools-and-three-tasks)
- [2. Task A — the model has it cold (no tool needed)](#2-task-a--the-model-has-it-cold-no-tool-needed)
- [3. Task B — data the model can't have (tool warranted)](#3-task-b--data-the-model-cant-have-tool-warranted)
- [4. Task C — the subtle one: bad at it, but will try anyway](#4-task-c--the-subtle-one-bad-at-it-but-will-try-anyway)
- [5. Takeaways](#5-takeaways)

## 1. Setup — two clean tools and three tasks

A `calculator` and a `current_time` tool, both plain typed functions with clean docstrings (we save the *design* walkthrough for [L0805](L0805_lecture.ipynb) — here they are just props). Three tasks span the decision: **A** the model owns cold, **B** the model can't have, **C** borderline arithmetic. Anchor model: **Claude Sonnet 4.6**. Live cells are gated on `LIVE`.

In [ ]:
from datetime import UTC, datetime
from zoneinfo import ZoneInfo

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

SONNET = "claude-sonnet-4-6"
LIVE = bool(get_settings().anthropic_api_key)


# Two clean tools. Their DESCRIPTIONS live in the docstrings now — bind_tools
# derives each tool definition (name + description + arg schema) from the typed
# function. We save the design walkthrough for L0805; here they are just props.
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the exact result.

    Use whenever the user asks for the result of a calculation.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # Safe ONLY because the whitelist above blocks names, attributes, and calls.
    return str(eval(expression))


def current_time(tz: str) -> str:
    """Return the current time in a given IANA timezone, e.g. 'Asia/Tokyo'.

    Use whenever the user asks for the current or real-time clock anywhere — the
    model has no clock of its own.
    """
    return datetime.now(UTC).astimezone(ZoneInfo(tz)).isoformat(timespec="seconds")


def show_turn(reply: AIMessage) -> None:
    """Print a model reply: its text, and any tool calls (name + args).

    Example output:
        text       'Let me look that up.'
        tool call  current_time(tz='Asia/Tokyo')  id=toolu_01...
    """
    if reply.text:
        print(f"  text       {reply.text!r}")
    for call in reply.tool_calls:
        args = ", ".join(f"{k}={v!r}" for k, v in call["args"].items())
        print(f"  tool call  {call['name']}({args})  id={call['id']}")


def called_tools(reply: AIMessage) -> list[str]:
    """Names of the tools the model proposed in this reply (possibly empty)."""
    return [call["name"] for call in reply.tool_calls]


TASK_A = "What is the capital of France?"  # model owns it cold -> NO tool
TASK_B = "What is the current time in Tokyo right now?"  # can't be memorized -> tool
TASK_C = "What is 18,374 multiplied by 92,431?"  # bad at it, but will try -> tool

if LIVE:
    model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=300)

print("setup ready (LIVE =", LIVE, ")")

[↑ Back to top](#top)

## 2. Task A — the model has it cold (no tool needed)

Run Task A model-alone: instant, free, correct. Then bind the calculator and run it again — the model usually answers *without* calling the tool. A bound tool is an **option, not an obligation**; here it would be pure overhead and a wrong-tool option to pick by mistake.

In [ ]:
if LIVE:
    alone = model.invoke([HumanMessage(TASK_A)])
    print("=== Task A, model-alone ===")
    show_turn(alone)

    withtool = model.bind_tools([calculator]).invoke([HumanMessage(TASK_A)])
    print("\n=== Task A, calculator bound ===")
    show_turn(withtool)
    print("tools called:", called_tools(withtool), "(expect none — a tool here is overhead)")
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 3. Task B — data the model can't have (tool warranted)

Run Task B model-alone: the model refuses, hedges (*'I don't have access to real-time data'*), or hallucinates a time. Then register `current_time`: it calls the tool and gets a real answer. This is the **'data the model can't have memorized'** signal.

In [ ]:
if LIVE:
    alone = model.invoke([HumanMessage(TASK_B)])
    print("=== Task B, model-alone (hedge / guess expected) ===")
    show_turn(alone)

    withtool = model.bind_tools([current_time]).invoke([HumanMessage(TASK_B)])
    print("\n=== Task B, current_time bound ===")
    show_turn(withtool)
    print("tools called:", called_tools(withtool), "(expect ['current_time'])")
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 4. Task C — the subtle one: bad at it, but will try anyway

Run Task C model-alone **three times**. The answers are often inconsistent across runs, and sometimes wrong — the model *will* attempt the arithmetic, confidently. Then register the calculator: a deterministic, correct answer. Task C is the subtle signal: the tool fills a gap the **designer** knows about, not one the model knows about.

In [ ]:
if LIVE:
    print("=== Task C, model-alone x3 (watch for drift) ===")
    for i in range(3):
        r = model.invoke([HumanMessage(TASK_C)])
        print(f"  run {i + 1}: {r.text.strip()[:80]}")

    print("\nground truth (Python):", 18374 * 92431)

    withtool = model.bind_tools([calculator]).invoke([HumanMessage(TASK_C)])
    print("\n=== Task C, calculator bound ===")
    show_turn(withtool)
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 5. Takeaways

- The four **'tool warranted'** signals (from [objectives.md](../../../../docs/origin/lesson_roadmaps/L08/objectives.md)): data the model can't memorize (B), precise computation it's bad at (C), side effects (foreshadow [L0809](L0809_lecture.ipynb)), verification against ground truth. Name each as it appears.
- The **'no tool needed'** case (A): the model already owns it reliably, and the round-trip isn't worth a marginal gain. A tool here wastes tokens and adds a wrong-tool failure mode.
- **Adding a tool is designing the agent's affordances** — an architectural decision. *More tools ≠ more capable agent* (reinforce [L01](../L01/L0101_intro.md) on context-window cost; don't re-teach).
- The deliverable is **one sentence defending each call** — naming *why* is the skill the [L0804 lab](L0804_lab_empty.ipynb) drills.
- Next: [L0805](L0805_lecture.ipynb) — once you've decided *yes, a tool*, the description decides whether the model uses it well.

[↑ Back to top](#top)